# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

ds_perf = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
df = ds_perf["train"].to_pandas()
print(df.shape)

(9841378, 30)


In [5]:
features = df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]].copy()

print(features.head())
print(features.shape)


   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  \
0               20           0          3.350000           NaN   
1                1           0          0.000000           NaN   
2              125           1          4.928000           NaN   
3                7           0          4.000000           NaN   
4               11           0          2.272727           NaN   

   ga4_engaged_sessions  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN  
(9841378, 5)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature notes: gsc_impressions and gsc_clicks are known at decision time — logged by Google Search Console for that same report_date, before the next day starts. gsc_avg_position is computed from that day's search data, also available same-day. ga4_sessions and ga4_engaged_sessions are logged by Google Analytics 4 for that day, available same-day, but show NaN for rows where a client has no GA4 connection — missing values are filled with 0 later using .fillna(0), since a missing GA4 value typically means no recorded activity that day, not an unknown value. Categorical flags like client_has_gsc and client_has_ga4 are booleans, used as-is without encoding.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I attacked my own feature set by testing what happens if the future label leaks in. Using "today's clicks" to predict "next day's clicks" pushed R² from an honest 0.311 to a leaky 0.377 — a real but smaller leak. Deliberately smuggling the future_clicks value itself in as a feature pushed R² to a fake, useless 1.0 — a clear giveaway of label leakage. The fix: drop any column derived from the future/target window, keeping only same-day, decision-time features (gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions), which restores the honest R² of 0.311.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded: fact_content_daily_performance_sample.parquet — the sealed June 2026 test month, deliberately held out and never touched while building features or tuning the label, to preserve an unbiased final evaluation. Excluded as features (kept only as context): report_date, client_hash_id, content_hash_id — these identify the row but aren't predictive signals, so they're used only for grouping/joining, never fed directly into the model.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.